In [69]:
import anthropic
from anthropic.types import Message
from dotenv import load_dotenv

load_dotenv()

def add_user_messages(messages, message):
    messages.append(
        {
            "role": "user", 
            "content": message.content if isinstance(message, Message) else message
        }
    )

def add_assistant_messages(messages, message):
    messages.append(
        {
            "role": "assistant",
            "content": message.content if isinstance(message, Message) else message
        }
    )
    
def chat(messages, stop_sequences=[], temperature=1.0, system=None, tools=None):
    client = anthropic.Anthropic()
    parameters = {
        "model": "claude-haiku-4-5",
        "max_tokens": 512,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    if tools:
        parameters["tools"] = tools

    if system:
        parameters["system"] = system

    response = client.messages.create(**parameters)
    return response


In [70]:
from datetime import datetime
from anthropic.types import ToolParam
from datetime import datetime, timedelta
import json

def get_current_datetime(format="%Y-%m-%d %H:%M:%S"):
    if not format:
        raise ValueError("Format string cannot be empty")
    return datetime.now().strftime(format)

def add_duration_to_datetime(
        datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


get_current_datetime_schema = ToolParam(
    {
        "name": "get_current_datetime",
        "description": "Retrieves the current date and time in the specified format. This tool is useful when you need to know the present date, time, or both. You can customize the output format using Python strftime syntax (e.g., '%Y-%m-%d' for just the date, '%H:%M:%S' for just the time). If no format is provided, it defaults to 'YYYY-MM-DD HH:MM:SS' format. The tool will raise an error if an empty format string is provided.",
        "input_schema": {
            "type": "object",
            "properties": {
            "format": {
                "type": "string",
                "description": "Python strftime format string to control the output format. Common examples: '%Y-%m-%d' (2024-05-27), '%H:%M:%S' (14:30:45), '%Y-%m-%d %H:%M:%S' (2024-05-27 14:30:45), '%A, %B %d, %Y' (Monday, May 27, 2024). Defaults to '%Y-%m-%d %H:%M:%S' if not provided. Cannot be an empty string.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
            },
            "required": []
        },
        "input_examples": [
            {
            "format": "%Y-%m-%d"
            },
            {
            "format": "%H:%M:%S"
            },
            {
            "format": "%A, %B %d, %Y at %I:%M %p"
            },
            {}
        ]
    }
)

add_duration_to_datetime_schema = {
  "name": "add_duration_to_datetime",
  "description": "Add or subtract a duration from a given datetime string and return the result in a formatted string.",
  "input_schema": {
    "type": "object",
    "properties": {
      "datetime_str": {
        "type": "string",
        "description": "The input datetime string to modify. Default format is 'YYYY-MM-DD' unless input_format is specified."
      },
      "duration": {
        "type": "integer",
        "description": "The amount of time to add or subtract. Can be positive or negative. Default is 0.",
        "default": 0
      },
      "unit": {
        "type": "string",
        "enum": ["seconds", "minutes", "hours", "days", "weeks", "months", "years"],
        "description": "The unit of time for the duration. Default is 'days'.",
        "default": "days"
      },
      "input_format": {
        "type": "string",
        "description": "Python strftime format string for parsing the input datetime_str. Default is '%Y-%m-%d'.",
        "default": "%Y-%m-%d"
      }
    },
    "required": ["datetime_str"]
  }
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Set a reminder with content for a specific timestamp.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The reminder content or message."
            },
            "timestamp": {
                "type": "string",
                "description": "The time when the reminder should trigger."
            }
        },
        "required": ["content", "timestamp"]
    }
}

In [71]:
def run_tool(tool_request): 
    try:
        tool_output = ""
        if(tool_request.name == "get_current_datetime"):
            tool_output = get_current_datetime(**tool_request.input)
        if(tool_request.name == "add_duration_to_datetime"):
            tool_output = add_duration_to_datetime(**tool_request.input)
        if(tool_request.name == "set_reminder"):
            tool_output = set_reminder(**tool_request.input)
        if(tool_output == ""):
            raise ValueError("tool output is empty")
        return {
            "type": "tool_result",
            "tool_use_id": tool_request.id,
            "content": json.dumps(tool_output),
            "is_error": False
        }
    except Exception as e:
        return {
            "type": "tool_result",
            "tool_use_id": tool_request.id,
            "content": f"Error: {e}",
            "is_error": True
        }
def run_tools(message):
    print("Message content:", message.content)
    tool_requests = [
        block for block in message.content if block.type == "tool_use"
    ]

    tool_responses = []

    for tool_request in tool_requests:
        tool_responses.append(run_tool(tool_request))

    return tool_responses


In [72]:
def run_conversation(messages):
    while True:
        response = chat(messages, tools=[get_current_datetime_schema, add_duration_to_datetime_schema, set_reminder_schema])
        add_assistant_messages(messages, response)

        print("Assistant response:", response)
        if response.stop_reason != "tool_use":
            print("Conversation stopped by model. Stop reason:", response.stop_reason)
            break

        tool_results = run_tools(response)
        add_user_messages(messages, tool_results)
        print("Tool results:", tool_results)
    return messages

In [73]:
messages = []
add_user_messages(messages, "Create reminder for appointment after 2 month")
run_conversation(messages)
messages

Assistant response: Message(id='msg_01GxqscMaW2Z8zmAbaDr72f2', container=None, content=[TextBlock(citations=None, text="I'll create a reminder for an appointment 2 months from now. First, let me get the current date and calculate the date 2 months from now.", type='text'), ToolUseBlock(id='toolu_01Fynvqsn3zxTJtMfLTrUys5', caller=DirectCaller(type='direct'), input={'format': '%Y-%m-%d %H:%M:%S'}, name='get_current_datetime', type='tool_use'), ToolUseBlock(id='toolu_01FcpJqdzFNWxNK5txK3FYGR', caller=DirectCaller(type='direct'), input={'datetime_str': '2024-12-19', 'duration': 2, 'unit': 'months'}, name='add_duration_to_datetime', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=1383, outpu

[{'role': 'user', 'content': 'Create reminder for appointment after 2 month'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll create a reminder for an appointment 2 months from now. First, let me get the current date and calculate the date 2 months from now.", type='text'),
   ToolUseBlock(id='toolu_01Fynvqsn3zxTJtMfLTrUys5', caller=DirectCaller(type='direct'), input={'format': '%Y-%m-%d %H:%M:%S'}, name='get_current_datetime', type='tool_use'),
   ToolUseBlock(id='toolu_01FcpJqdzFNWxNK5txK3FYGR', caller=DirectCaller(type='direct'), input={'datetime_str': '2024-12-19', 'duration': 2, 'unit': 'months'}, name='add_duration_to_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01Fynvqsn3zxTJtMfLTrUys5',
    'content': '"2026-05-28 17:39:16"',
    'is_error': False},
   {'type': 'tool_result',
    'tool_use_id': 'toolu_01FcpJqdzFNWxNK5txK3FYGR',
    'content': '"Wednesday, February 19, 2025 12:00:00 AM